In [1]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Column coverage

In [2]:
parquet_path = Path("Geo/merged_all_geo.parquet")
output_path = Path("Final/column_coverage.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(parquet_path)

column_coverage = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "non_null_count": [df[col].notna().sum() for col in df.columns],
    "total_count": len(df),
})

column_coverage["coverage_pct"] = (
    column_coverage["non_null_count"] / column_coverage["total_count"] * 100
)

column_coverage.to_csv(output_path, index=False)
print(len(df))
print(f"Saved: {output_path}")
print(column_coverage)

3760265
Saved: Final\column_coverage.csv
                      column    dtype  non_null_count  total_count  \
0     acceptedScientificName   string         3760252      3760265   
1           acceptedTaxonKey    Int64         2400481      3760265   
2                    aphiaID    Int64         3760265      3760265   
3            acceptedAphiaID    Int64         3760265      3760265   
4                areaSampled  float64          342014      3760265   
..                       ...      ...             ...          ...   
66          trawlingDistance  float64               0      3760265   
67                      Keep      str         3288475      3760265   
68                  isMarine  boolean         3758864      3760265   
69                isBrackish  boolean         3026394      3760265   
70  geomorphologicalCategory      str         3760265      3760265   

    coverage_pct  
0      99.999654  
1      63.838081  
2     100.000000  
3     100.000000  
4       9.095476  
..  

# Fixes

## Remove redundant & empty columns

In [3]:
parquet_path = Path("Geo/merged_all_geo.parquet")
df = pd.read_parquet(parquet_path)

# Remove specified columns and save to Final directory
columns_to_remove = ["trawlingDistance", "depthA", "depthB", "Keep"]
final_df = df.drop(columns=[col for col in columns_to_remove if col in df.columns])

final_path = Path("Final/merged_all_final.parquet")
final_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(final_df.shape)

Saved: Final\merged_all_final.parquet
(3760265, 67)


## Fill in flagged column

In [4]:
# Update the 'flagged' column: set to True where 'flaggedComment' is not null
final_df['flagged'] = final_df['flagged'] | final_df['flaggedComment'].notna()

# Save the updated dataframe
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")

Saved: Final\merged_all_final.parquet


## Check for mojibake issues

In [5]:
# Load the final parquet and check for mojibake issues
final_df = pd.read_parquet(final_path)

mojibake_patterns = [
    "�",      # replacement character
    "â€",     # common UTF-8/Windows-1252 mojibake prefix
    "â€™",
    "â€œ",
    "â€�",
    "â€ž",
    "ï¿½",     # replacement char rendered as mojibake
    "Ã",
    "Â",
]

# Check string columns for mojibake indicators
string_cols = final_df.select_dtypes(include=['string', 'object']).columns
mojibake_found = False

for col in string_cols:
    series = final_df[col].astype('string[python]')
    
    mojibake_mask = pd.Series(False, index=series.index)
    for pattern in mojibake_patterns:
        mojibake_mask = mojibake_mask | series.str.contains(pattern, na=False, regex=False)
    
    if mojibake_mask.any():
        mojibake_found = True
        print(f"\n--- Mojibake detected in '{col}' ({mojibake_mask.sum()} rows) ---")
        sample = final_df.loc[mojibake_mask, col].drop_duplicates().head(10)
        print(sample.values)

if not mojibake_found:
    print("No mojibake issues detected in Final/merged_all_final.parquet")


--- Mojibake detected in 'eventRemarks' (1 rows) ---
<ArrowStringArray>
['RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 âˆšÃ¨17 T1']
Length: 1, dtype: string

--- Mojibake detected in 'verbatimID' (2 rows) ---
<ArrowStringArray>
['1764601749653-eu09oh2xu|185Ã', '1764601749653-eu09oh2xu|185Â']
Length: 2, dtype: string

--- Mojibake detected in 'verbatimScientificName' (200 rows) ---
<ArrowStringArray>
[                            'BonelliidaeÂ',
       'Micronephthys minuta (Th�el, 1879)',
     'Phyllodoce groenlandica �rsted, 1842',
   'Pectinaria granulata (Linnaeus, 1767)�',
              'Pholoe baltica �rsted, 1843',
         'Nephtys longosetosa �rsted, 1842',
 'Cirratulus cirratus (O. F. M�ller, 1776)',
       'Cylichna gouldii (Couthouy, 1839)�',
   'Scoletoma fragilis (O.F. M�ller, 1776)',
         'Nicolea zostericola �rsted, 1844']
Length: 10, dtype: string


In [6]:
# Check for mojibake (encoding issues) in string columns and repair recoverable text
mojibake_patterns = [
    "�",      # replacement character
    "â€",     # common UTF-8/Windows-1252 mojibake prefix
    "â€™",
    "â€œ",
    "â€�",
    "â€ž",
    "ï¿½",     # replacement char rendered as mojibake
    "Ã",
    "Â",
]

# Try to recover strings that were decoded with the wrong encoding
def fix_mojibake_text(value):
    if pd.isna(value) or not isinstance(value, str):
        return value

    for source_encoding in ("cp1252", "latin1"):
        try:
            fixed = value.encode(source_encoding).decode("utf-8")
            if fixed != value:
                return fixed
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass

    return value

# Check string columns for common mojibake indicators
string_cols = final_df.select_dtypes(include=['string', 'object']).columns
repaired_df = final_df.copy()
unrecoverable_examples = []

for col in string_cols:
    # Use Python-backed strings to avoid Arrow regex limitations
    series = repaired_df[col].astype('string[python]')

    mojibake_mask = pd.Series(False, index=series.index)
    for pattern in mojibake_patterns:
        mojibake_mask = mojibake_mask | series.str.contains(pattern, na=False, regex=False)

    if mojibake_mask.any():
        sample_original = repaired_df.loc[mojibake_mask, col].drop_duplicates().head(10)
        sample_corrected = sample_original.map(fix_mojibake_text)
        preview_df = pd.DataFrame({
            "original": sample_original.values,
            "corrected": sample_corrected.values,
        })
        preview_df = preview_df[preview_df["original"] != preview_df["corrected"]]

        if not preview_df.empty:
            print(f"\n--- Potential mojibake in '{col}' ({mojibake_mask.sum()} rows) ---")
            print(preview_df.to_string(index=False))

        unrecoverable_mask = sample_original == sample_corrected
        if unrecoverable_mask.any():
            unrecoverable_df = pd.DataFrame({
                "column": col,
                "value": sample_original[unrecoverable_mask].values,
            })
            unrecoverable_examples.append(unrecoverable_df)

    repaired_df[col] = repaired_df[col].map(fix_mojibake_text)

repaired_output_path = Path("Final/merged_all_final.parquet")
repaired_output_path.parent.mkdir(parents=True, exist_ok=True)
repaired_df.to_parquet(repaired_output_path, index=False)
print(f"\nSaved repaired parquet: {repaired_output_path}")

if unrecoverable_examples:
    unrecoverable_df = pd.concat(unrecoverable_examples, ignore_index=True).drop_duplicates()
    print("\n--- Potential entries not recovered by the repair pass ---")
    print(unrecoverable_df.head(20).to_string(index=False))
else:
    print("\nNo unrecoverable mojibake-like entries were found in the sampled values.")

print("\nMojibake repair complete!")


--- Potential mojibake in 'eventRemarks' (1 rows) ---
                                                 original                                              corrected
RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 âˆšÃ¨17 T1 RV Akademik M.A. Lavrentyev _cruise 82_st.BS18 √è17 T1

Saved repaired parquet: Final\merged_all_final.parquet

--- Potential entries not recovered by the repair pass ---
                column                                    value
            verbatimID             1764601749653-eu09oh2xu|185Ã
            verbatimID             1764601749653-eu09oh2xu|185Â
verbatimScientificName                             BonelliidaeÂ
verbatimScientificName       Micronephthys minuta (Th�el, 1879)
verbatimScientificName     Phyllodoce groenlandica �rsted, 1842
verbatimScientificName   Pectinaria granulata (Linnaeus, 1767)�
verbatimScientificName              Pholoe baltica �rsted, 1843
verbatimScientificName         Nephtys longosetosa �rsted, 1842
verbatimScientificName Cirr

## Remove -999 values from individualCount

In [7]:
# Load the final parquet and replace -999 values with NaN in individualCount
final_df = pd.read_parquet(final_path)

# Replace -999 with NaN in individualCount column
final_df['individualCount'] = final_df['individualCount'].replace(-999.0, pd.NA)

# Save the updated dataframe
final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")
print(f"Replaced -999 values with NaN in individualCount")

Saved: Final\merged_all_final.parquet
Replaced -999 values with NaN in individualCount


## Check for other sentinel values that should be replaced with NaN

In [8]:
# Scan for common sentinel values that may need to be converted to NA/NaN
sentinel_values = [-999, -999.0, -9999, -9999.0, 999, 999.0, 9999, 9999.0, 99999, 99999.0]
sentinel_strings = {"-999", "-999.0", "-9999", "-9999.0", "999", "999.0", "9999", "9999.0", "99999", "99999.0"}

summary = []

for col in final_df.columns:
    s = final_df[col]

    if pd.api.types.is_numeric_dtype(s):
        for sentinel in sentinel_values:
            mask = s.eq(sentinel)
            if mask.any():
                summary.append({
                    "column": col,
                    "sentinel": sentinel,
                    "count": int(mask.sum()),
                    "sample": s.loc[mask].head(3).tolist(),
                })

    elif pd.api.types.is_string_dtype(s) or s.dtype == "object":
        ss = s.astype("string[python]")
        for sentinel in sentinel_strings:
            mask = ss.eq(sentinel)
            if mask.any():
                summary.append({
                    "column": col,
                    "sentinel": sentinel,
                    "count": int(mask.sum()),
                    "sample": ss.loc[mask].head(3).tolist(),
                })

summary_df = pd.DataFrame(summary).sort_values(["count", "column"], ascending=[False, True])

if summary_df.empty:
    print("No common sentinel values found.")
else:
    print("Common sentinel values found:")
    print(summary_df.to_string(index=False))

    # Optional: show columns with any sentinel hits
    print("\nColumns with sentinel hits:")
    print(summary_df["column"].drop_duplicates().tolist())

Common sentinel values found:
    column sentinel  count                   sample
gebcoDepth     -999     44 [-999.0, -999.0, -999.0]
gebcoDepth   -999.0     44 [-999.0, -999.0, -999.0]
  depthA_m      999     28    [999.0, 999.0, 999.0]
  depthA_m    999.0     28    [999.0, 999.0, 999.0]
  depthB_m      999     16    [999.0, 999.0, 999.0]
  depthB_m    999.0     16    [999.0, 999.0, 999.0]
   eventID      999     15          [999, 999, 999]
   eventID     9999      3       [9999, 9999, 9999]
verbatimID      999      2               [999, 999]
  sourceID    99999      1                  [99999]
  sourceID     9999      1                   [9999]
  sourceID      999      1                    [999]

Columns with sentinel hits:
['gebcoDepth', 'depthA_m', 'depthB_m', 'eventID', 'verbatimID', 'sourceID']


## Fill in missing acceptedScientificName

In [9]:
mask = final_df["acceptedScientificName"].isna() & final_df["acceptedAphiaID"].notna()

unique_accepted_aphia_ids = (
    final_df.loc[mask, "acceptedAphiaID"]
    .drop_duplicates()
    .sort_values()
)

print(unique_accepted_aphia_ids.to_string(index=False))

 128497
 182832
 182904
 206284
 741024
1584794


In [10]:
# Fill missing acceptedScientificName and validAuthority from acceptedAphiaID

aphia_map = {
    101070: {"acceptedScientificName": "Echinoderidae", "validAuthority": "Carus, 1885"},
    128497: {"acceptedScientificName": "Distichoptilum", "validAuthority": "Verrill, 1882"},
    182832: {"acceptedScientificName": "Nellobia", "validAuthority": "Fisher, 1946"},
    182904: {"acceptedScientificName": "Vitjazema", "validAuthority": "Zenkevitch, 1958"},
    206284: {"acceptedScientificName": "Zoanthus", "validAuthority": "Lamarck, 1801"},
    741024: {"acceptedScientificName": "Antheidae", "validAuthority": "Gosse, 1860"},
    1584794: {"acceptedScientificName": "Epomidiostomum", "validAuthority": "Skryabin, 1915"},
}

final_df = pd.read_parquet(final_path)

for aphia_id, values in aphia_map.items():
    mask = final_df["acceptedAphiaID"].eq(aphia_id)
    if "acceptedScientificName" in final_df.columns:
        final_df.loc[mask & final_df["acceptedScientificName"].isna(), "acceptedScientificName"] = values["acceptedScientificName"]
    if "validAuthority" in final_df.columns:
        final_df.loc[mask & final_df["validAuthority"].isna(), "validAuthority"] = values["validAuthority"]

final_df.to_parquet(final_path, index=False)
print(f"Saved: {final_path}")

Saved: Final\merged_all_final.parquet


# Get number of taxa per station

In [11]:
stations = pd.read_parquet("Final/merged_all_final.parquet")

cols = [
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "datasetKey",
    "datasetTitle",
    "sourceDatabase",
    "year",
    "gebcoDepth",
    "startDayOfYear",
    "gear", 
]
all_stations = stations[cols].drop_duplicates()

total_stations = len(all_stations)
print(f"Total unique stations: {total_stations}")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

all_stations

Total unique stations: 233723


,decimalLatitudeStart,decimalLongitudeStart,datasetKey,datasetTitle,sourceDatabase,year,gebcoDepth,startDayOfYear,gear
0,70.001883,-126.254916,83eac1c3-e549-4eba-9dad-01d81ba8b94c,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2022,-212.0,230,NaN
1,63.718060,-68.518610,8dbd0669-92d1-41ef-a5db-527bc6aa1ec9,Biodiversity of benthic assemblages on the Arc...,OBIS,1969,4.0,246,NaN
2,53.292600,-169.778670,f5a4799e-dc24-4807-89d9-01da47d52e3b,"NOAA, Deep-Sea Coral Research and Technology P...",OBIS,2023,-792.0,204,NaN
3,55.740002,-162.823334,89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrat...,OBIS,1982,-57.0,304,NaN
4,72.351167,-119.073333,55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19,DFO Arctic Region Biodiversity of the Benthic ...,OBIS,2024,-77.0,238,NaN
...,...,...,...,...,...,...,...,...,...
3759538,67.867250,-54.968250,DISKO WEST,NaN,AU,2009,-35.0,145,Van Veen
3759601,67.972950,-54.819167,DISKO WEST,NaN,AU,2009,-26.0,145,Van Veen
3759704,67.972950,-54.819150,DISKO WEST,NaN,AU,2009,-26.0,145,Van Veen
3759840,68.125200,-54.964817,DISKO WEST,NaN,AU,2009,-54.0,146,Van Veen


In [12]:
# Count unique aphiaID per station row and overwrite Final/all_stations.parquet

count_col = "uniqueAphiaIDCount"

aphia_counts = (
    stations.groupby(cols, dropna=False)["acceptedAphiaID"]
    .nunique()
    .reset_index(name=count_col)
)

if count_col in all_stations.columns:
    all_stations = all_stations.drop(columns=[count_col])

all_stations = all_stations.merge(aphia_counts, on=cols, how="left")
all_stations[count_col] = all_stations[count_col].fillna(0).astype("Int64")

all_stations.to_parquet("Final/all_stations.parquet", index=False)

print(f"Saved: Final/all_stations.parquet")
print(all_stations[[*cols, count_col]].head())

Saved: Final/all_stations.parquet
   decimalLatitudeStart  decimalLongitudeStart  \
0             70.001883            -126.254916   
1             63.718060             -68.518610   
2             53.292600            -169.778670   
3             55.740002            -162.823334   
4             72.351167            -119.073333   

                             datasetKey  \
0  83eac1c3-e549-4eba-9dad-01d81ba8b94c   
1  8dbd0669-92d1-41ef-a5db-527bc6aa1ec9   
2  f5a4799e-dc24-4807-89d9-01da47d52e3b   
3  89e23fc8-3f61-4480-9de3-358fe6eefe0b   
4  55ff4bf2-b3d2-49bf-8749-7c41fb2c0c19   

                                        datasetTitle sourceDatabase  year  \
0  DFO Arctic Region Biodiversity of the Benthic ...           OBIS  2022   
1  Biodiversity of benthic assemblages on the Arc...           OBIS  1969   
2  NOAA, Deep-Sea Coral Research and Technology P...           OBIS  2023   
3  National Museum of Natural History Invertebrat...           OBIS  1982   
4  DFO Arctic Region 

# Datasets

In [5]:
datasets_path = Path("Final/datasets.xlsx")
datasets_path.parent.mkdir(parents=True, exist_ok=True)

datasets_df = (
    pd.read_parquet("Final/merged_all_final.parquet", columns=["datasetTitle", "datasetKey", "sourceDatabase"])
    .dropna()
    .groupby(["datasetTitle", "datasetKey", "sourceDatabase"], as_index=False)
    .size()
    .rename(columns={"size": "n_records"})
    .sort_values(["datasetTitle", "sourceDatabase"])
)

datasets_df.to_excel(datasets_path, index=False)
print(f"Saved: {datasets_path}")
print(datasets_df.shape)
print(datasets_df.head())

Saved: Final\datasets.xlsx
(436, 4)
                                        datasetTitle  \
0  1776-1997 Paul F. Clark North East Atlantic Cr...   
1  A Biotic Database of Indo-Pacific Marine Mollusks   
2  A new deepwater species of Calliopiidae, Halir...   
3  A new genus and five new species of the subfam...   
4  A revision of the genus Calcigorgia (Cnidaria,...   

                             datasetKey sourceDatabase  n_records  
0  67100d5f-8831-4a67-a553-fad388d96630           OBIS         54  
1  8bd1c7c0-717d-4a4e-8b1c-9ff3c5698067           OBIS         11  
2  9a23e0ce-ffdb-4ec1-9796-403ffb9ada90           GBIF          1  
3  56e22b1b-6708-4761-9a11-873c77c3b453           GBIF          5  
4  a9fc1eb5-487b-4f58-9c68-954498dd39a3           GBIF         15  
